# L3 — Set up Resources: Registry, Gateway & Cognito

## What You'll Learn

- How to create an AgentCore Registry as a centralized catalog for specialist agents
- How to enable CloudWatch Transaction Search for end-to-end agent observability
- How to authorize the tool Gateway with a Cognito M2M (client_credentials) service token
- How to expose registry discovery to the Harness through a second, IAM-auth Gateway
- How to provision test users and publish all resource IDs to SSM for downstream notebooks
- How to store Datadog credentials in SSM for agent telemetry

## Overview

Before deploying any specialist agents, AnyCompany's multi-agent system needs shared infrastructure: a Registry where agents publish themselves, and Gateways that route requests. This notebook creates the Registry, a **tool** Gateway (`anycompany-agent-gateway`) for specialist tool calls, authorized by a Cognito **M2M (client_credentials)** service token, and a second **IAM-auth** Gateway (`anycompany-harness-gateway`) that exposes registry discovery to the Harness — along with the Cognito User Pool that authenticates chatbot users.

The Registry starts empty. Notebooks 2 and 3 each deploy a specialist agent and register it here. Once registered, the Harness (Notebook 4) discovers them dynamically — no hardcoded endpoints. All resource IDs are stored in SSM Parameter Store so every subsequent notebook can look them up without re-creating anything.

## Architecture

```
  ┌──────────────────────────────────────────────────────────────────┐
  │  Created in this notebook                                        │
  │                                                                  │
  │  AgentCore Registry  ◄── specialist agents register here         │
  │  (anycompany-agent-registry, auto-approval enabled)              │
  │                                                                  │
  │  AgentCore Gateway — tools  ◄── specialist tool calls            │
  │  (anycompany-agent-gateway, MCP, Cognito M2M client_credentials) │
  │     └─ IAM Role → order + refund Lambda tools                    │
  │                                                                  │
  │  AgentCore Gateway — registry  ◄── Harness discovery/invoke      │
  │  (anycompany-harness-gateway, MCP, AWS_IAM auth)                 │
  │     └─ IAM Role → anycompany_registry_tools Lambda               │
  │                                                                  │
  │  Cognito User Pool                                               │
  │  ├── users: gold / silver / bronze  (chatbot identity)           │
  │  ├── resource server: anycompany-gateway  (scope: invoke)        │
  │  ├── M2M app client (client_credentials) → gateway service token │
  │  └── hosted domain → OAuth2 /oauth2/token endpoint               │
  │                                                                  │
  │  Datadog Credentials (validated + stored in SSM)                 │
  │  ├── dd_api_key   → trace-agent sidecar intake                   │
  │  ├── dd_app_key   → AI Guard + API queries                       │
  │  └── dd_site      → Datadog region (datadoghq.com)               │
  └──────────────────────────────────────────────────────────────────┘
                    │
                    ▼
              SSM Parameter Store
              /anycompany/agentcore/...
              (consumed by Notebooks 2–4, L4, L5, and L6)
```

## Steps

1. Enable CloudWatch Transaction Search
2. Create the AgentCore Registry
3. Create the Cognito User Pool, test users, and the M2M gateway service client
4. Create the AgentCore Gateway
5. Create the Registry Discovery Tool (IAM Gateway + Lambda)
6. Save all resource IDs to SSM Parameter Store
7. Store Datadog credentials in SSM for agent observability

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- AWS credentials configured with permissions for AgentCore, Cognito, IAM, Lambda, CloudWatch, X-Ray, and SSM
- `boto3` (installed in the first code cell)
- L2 notebook completed — `model_id` and `guardrail_id` must be in SSM before running L3 agents
- This notebook must run before Notebooks 2, 3, and 4

Install required packages.

In [1]:
%pip install --quiet --upgrade boto3 datadog-api-client==2.31.0

Note: you may need to restart the kernel to use updated packages.


Import libraries, set the region, and initialise AWS clients.

In [2]:
import boto3
import json
import time
import botocore
import os

# --- Configuration ---
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1") # Change to your preferred region
REGISTRY_NAME = "anycompany-agent-registry"
GATEWAY_NAME = "anycompany-agent-gateway"

# Cognito User Pool config (for Gateway OAuth auth)
# These will be created in Notebook 6, but set them here if you already have them
COGNITO_USER_POOL_ID = os.environ.get("COGNITO_USER_POOL_ID", "")  # e.g. us-east-1_xxxxxxx
COGNITO_CLIENT_ID = os.environ.get("COGNITO_CLIENT_ID", "")  # App client ID

# Clients
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
agent_registry = boto3.client("agent-registry-control", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)

ACCOUNT_ID = sts.get_caller_identity()["Account"]
print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")

Account: 359894365202
Region:  us-west-2


## Step 1: Enable CloudWatch Transaction Search (Optional)

**Optional — enable only if you want the Harness's traces searchable in the CloudWatch / X-Ray console.**

Transaction Search routes the AgentCore **Harness's** X-Ray spans into CloudWatch Logs so they are
searchable there. It is a CloudWatch-native *trace search* feature — it does **not** affect application
logs or metrics, which flow to CloudWatch regardless of this step.

You do **not** need this step to see Harness traces in Datadog: the Harness exports its traces to Datadog
directly via OTLP (Notebook 4, Step 5), landing in both APM and Agent Observability. The specialist agents
send their traces to Datadog via the trace-agent sidecar (Notebooks 2–3).

- **Run this step** if you also want CloudWatch / X-Ray-native trace search for the Harness.
- **Skip it** if you are observing everything in Datadog (the default for this track) — no logs or metrics
  are lost by skipping; you only forgo the CloudWatch-side span-search view.

Create the X-Ray resource policy, set trace destination to CloudWatch Logs, enable Application Signals, and verify status.

In [3]:
# Enable CloudWatch Transaction Search for AgentCore Observability
logs_client = boto3.client("logs", region_name=REGION)
xray = boto3.client("xray", region_name=REGION)
app_signals = boto3.client("application-signals", region_name=REGION)

# 1. Resource policy: allow X-Ray to write spans to CloudWatch Logs
logs_client.put_resource_policy(
    policyName="TransactionSearchXRayAccess",
    policyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Sid": "TransactionSearchXRayAccess",
            "Effect": "Allow",
            "Principal": {"Service": "xray.amazonaws.com"},
            "Action": "logs:PutLogEvents",
            "Resource": [
                f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:aws/spans:*",
                f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/application-signals/data:*"
            ],
            "Condition": {
                "ArnLike": {"aws:SourceArn": f"arn:aws:xray:{REGION}:{ACCOUNT_ID}:*"},
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID}
            }
        }]
    })
)
print("✓ Resource policy created")

# 2. Route trace segments to CloudWatch Logs
try:
    xray.update_trace_segment_destination(Destination="CloudWatchLogs")
    print("✓ Trace destination set to CloudWatchLogs")
except xray.exceptions.InvalidRequestException:
    print("✓ Trace destination already set to CloudWatchLogs")

# 3. Index 1% of spans (free tier)
xray.update_indexing_rule(Name="Default", Rule={"Probabilistic": {"DesiredSamplingPercentage": 1}})
print("✓ Indexing rule set to 1%")

# 4. Enable Application Signals
app_signals.start_discovery()
print("✓ Application Signals enabled")

# 5. Verify
resp = xray.get_trace_segment_destination()
status = resp['Status']
print(f"\nTransaction Search: {resp['Destination']} ({status})")
if status == 'PENDING':
    print("\n⏳ Status is PENDING. This can take up to 10 minutes.")
    print("   You can proceed with the next steps — Transaction Search will be ready by the time you invoke agents.")
elif status == 'ACTIVE':
    print("\n✓ Transaction Search is ACTIVE")

✓ Resource policy created
✓ Trace destination already set to CloudWatchLogs
✓ Indexing rule set to 1%
✓ Application Signals enabled

Transaction Search: CloudWatchLogs (ACTIVE)

✓ Transaction Search is ACTIVE


## Step 2: Create the AgentCore Registry

Centralized catalog where specialist agents publish their metadata. Auto-approval is enabled so agents become discoverable immediately on registration.

NOTE: It takes typically 3-5 minutes to create registry

In [4]:
# Check if registry already exists before attempting to create
registries = agent_registry.list_registries()
registry_match = None
for reg in registries.get("registries", []):
    if reg.get("name") == REGISTRY_NAME:
        registry_match = reg
        break

if registry_match:
    REGISTRY_ARN = registry_match["registryArn"]
    REGISTRY_ID = REGISTRY_ARN.split("/")[-1]
    print(f"Registry '{REGISTRY_NAME}' already exists.")
    print(f"  ID:     {REGISTRY_ID}")
    print(f"  ARN:    {REGISTRY_ARN}")
    print(f"  Status: {registry_match.get('status')}")
else:
    print(f"Creating registry: {REGISTRY_NAME}\n")
    resp = agent_registry.create_registry(
        name=REGISTRY_NAME,
        description="AnyCompany agent registry — centralized catalog for order, refund, and vision agents.",
        approvalConfiguration={"autoApprovalRules": ["APPROVE_ALL"]},
    )
    REGISTRY_ARN = resp["registryArn"]
    REGISTRY_ID = REGISTRY_ARN.split("/")[-1]
    print(f"Created registry: {REGISTRY_NAME} (ID: {REGISTRY_ID})")
    print(f"  ARN:           {REGISTRY_ARN}")
    print(f"  Auto-approval: Enabled")


Creating registry: anycompany-agent-registry

Created registry: anycompany-agent-registry (ID: XCtvGlP9E1u5ZoYr)
  ARN:           arn:aws:agent-registry:us-west-2:359894365202:registry/XCtvGlP9E1u5ZoYr
  Auto-approval: Enabled


Poll until the Registry reaches READY status.

In [5]:
# Poll until the registry transitions from CREATING to READY
import time as _time

print("Waiting for registry to be ready...")
_reg_start = _time.time()
_poll = 0

while True:
    r = agent_registry.get_registry(registryId=REGISTRY_ID)
    _elapsed = _time.time() - _reg_start
    _poll += 1
    if r["status"] == "READY":
        print(f"Registry is READY after {_elapsed:.1f}s ({_poll} poll(s))")
        break
    print(f"  [{_elapsed:5.1f}s] Status: {r['status']} - waiting... (poll #{_poll})")
    time.sleep(3)

print(f"\nRegistry ID:  {REGISTRY_ID}")
print(f"Registry ARN: {REGISTRY_ARN}")

Waiting for registry to be ready...
  [  0.0s] Status: CREATING - waiting... (poll #1)
  [  3.1s] Status: CREATING - waiting... (poll #2)
  [  6.1s] Status: CREATING - waiting... (poll #3)
  [  9.2s] Status: CREATING - waiting... (poll #4)
  [ 12.2s] Status: CREATING - waiting... (poll #5)
  [ 15.3s] Status: CREATING - waiting... (poll #6)
  [ 18.3s] Status: CREATING - waiting... (poll #7)
  [ 21.3s] Status: CREATING - waiting... (poll #8)
  [ 24.4s] Status: CREATING - waiting... (poll #9)
  [ 27.4s] Status: CREATING - waiting... (poll #10)
  [ 30.5s] Status: CREATING - waiting... (poll #11)
  [ 33.5s] Status: CREATING - waiting... (poll #12)
  [ 36.5s] Status: CREATING - waiting... (poll #13)
  [ 39.6s] Status: CREATING - waiting... (poll #14)
  [ 42.6s] Status: CREATING - waiting... (poll #15)
  [ 45.7s] Status: CREATING - waiting... (poll #16)
  [ 48.7s] Status: CREATING - waiting... (poll #17)
  [ 51.7s] Status: CREATING - waiting... (poll #18)
  [ 54.8s] Status: CREATING - waiting

Confirm the Registry is empty — specialist agents will register in Notebooks 2 and 3.

In [6]:
records_response = agent_registry.list_registry_records(registryId=REGISTRY_ID)
records = records_response.get("registryRecords", [])
print(f"Registry records: {len(records)}")
if len(records) == 0:
    print("Registry is empty — specialist agents will register in Notebooks 2-4.")
else:
    for r in records:
        print(f"  - {r.get('name', 'unknown')} ({r.get('status', 'unknown')})")

Registry records: 0
Registry is empty — specialist agents will register in Notebooks 2-4.


## Step 3: Create the Cognito User Pool

Authenticates chatbot users. The Gateway validates JWT tokens from this pool via OIDC discovery.

Create the Cognito User Pool and app client, or reuse existing ones. 

The Cognito pool configuration is for demonstration purposes only and should not be used for production.

In [7]:
cognito = boto3.client("cognito-idp", region_name=REGION)
POOL_NAME = "AnyCompanyAgentPool"

if not COGNITO_USER_POOL_ID:
    pool_resp = cognito.create_user_pool(
        PoolName=POOL_NAME,
        Policies={
            "PasswordPolicy": {
                "MinimumLength": 12,
                "RequireUppercase": True,
                "RequireLowercase": True,
                "RequireNumbers": True,
                "RequireSymbols": True,
            }
        },
        AutoVerifiedAttributes=["email"],
        Schema=[
            {"Name": "email", "Required": True, "Mutable": True},
            {"Name": "customer_id", "AttributeDataType": "String", "Mutable": True, "StringAttributeConstraints": {"MinLength": "1", "MaxLength": "50"}},
            {"Name": "tier", "AttributeDataType": "String", "Mutable": True, "StringAttributeConstraints": {"MinLength": "1", "MaxLength": "20"}},
        ],
    )
    COGNITO_USER_POOL_ID = pool_resp["UserPool"]["Id"]
    print(f"Created User Pool: {COGNITO_USER_POOL_ID}")

    client_resp = cognito.create_user_pool_client(
        UserPoolId=COGNITO_USER_POOL_ID,
        ClientName="AnyCompanyChatbot",
        ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
        GenerateSecret=False,
    )
    COGNITO_CLIENT_ID = client_resp["UserPoolClient"]["ClientId"]
    print(f"Created App Client: {COGNITO_CLIENT_ID}")
else:
    print(f"Using existing pool: {COGNITO_USER_POOL_ID}")
    if not COGNITO_CLIENT_ID:
        clients = cognito.list_user_pool_clients(UserPoolId=COGNITO_USER_POOL_ID, MaxResults=10)
        COGNITO_CLIENT_ID = clients["UserPoolClients"][0]["ClientId"]
    print(f"Using existing client: {COGNITO_CLIENT_ID}")

print(f"\nCognito User Pool ID: {COGNITO_USER_POOL_ID}")
print(f"Cognito Client ID:    {COGNITO_CLIENT_ID}")

# ── M2M (service) auth for the Gateway ────────────────────────────────────────
# AgentCore Gateway (CUSTOM_JWT) requires an OAuth2 access token that carries a
# real scope. USER_PASSWORD_AUTH ID tokens have no scope, so specialist tool
# calls get 403 insufficient_scope. Add a Cognito resource server + a
# machine-to-machine (client_credentials) app client whose access token carries
# `<resource-server>/invoke`. The specialist agents use THIS token to call the
# gateway; the tiered users above stay for chatbot customer identity (customer_id).
RESOURCE_SERVER_ID = "anycompany-gateway"
GATEWAY_SCOPE_NAME = "invoke"
GATEWAY_SCOPE = f"{RESOURCE_SERVER_ID}/{GATEWAY_SCOPE_NAME}"
M2M_CLIENT_NAME = "AnyCompanyGatewayM2M"

# 1) Resource server defines the custom scope (idempotent).
try:
    cognito.describe_resource_server(UserPoolId=COGNITO_USER_POOL_ID, Identifier=RESOURCE_SERVER_ID)
    print(f"Resource server exists: {RESOURCE_SERVER_ID}")
except cognito.exceptions.ResourceNotFoundException:
    cognito.create_resource_server(
        UserPoolId=COGNITO_USER_POOL_ID,
        Identifier=RESOURCE_SERVER_ID,
        Name="AnyCompany Gateway",
        Scopes=[{"ScopeName": GATEWAY_SCOPE_NAME, "ScopeDescription": "Invoke AnyCompany gateway tools"}],
    )
    print(f"Created resource server: {RESOURCE_SERVER_ID} (scope {GATEWAY_SCOPE})")

# 2) Hosted-UI domain is required for the OAuth2 token endpoint. Reuse if present.
_pool = cognito.describe_user_pool(UserPoolId=COGNITO_USER_POOL_ID)["UserPool"]
# Domain prefix must be POOL-UNIQUE: nb1 creates a fresh user pool on each fresh-kernel
# run and Cognito domain prefixes are GLOBALLY unique + bound to one pool. An
# account-wide prefix would stay bound to a PRIOR pool, so the token endpoint would
# authenticate against the wrong pool and reject this pool's M2M client (400
# invalid_client). Derive it from the pool id so each pool gets its own domain.
DOMAIN_PREFIX = _pool.get("Domain") or f"anycompany-agentcore-{COGNITO_USER_POOL_ID.split('_')[-1].lower()}"
if not _pool.get("Domain"):
    try:
        cognito.create_user_pool_domain(Domain=DOMAIN_PREFIX, UserPoolId=COGNITO_USER_POOL_ID)
        print(f"Created Cognito domain: {DOMAIN_PREFIX}")
    except cognito.exceptions.InvalidParameterException as e:
        print(f"Domain create note ({DOMAIN_PREFIX}): {e}")
else:
    print(f"Using existing Cognito domain: {DOMAIN_PREFIX}")
# Wait for the domain to be ACTIVE — the OAuth token endpoint 400s until it is.
for _ in range(30):
    _dstatus = cognito.describe_user_pool_domain(Domain=DOMAIN_PREFIX)["DomainDescription"].get("Status")
    if _dstatus == "ACTIVE":
        break
    print(f"  domain status: {_dstatus} \u2014 waiting...")
    time.sleep(10)
GATEWAY_TOKEN_ENDPOINT = f"https://{DOMAIN_PREFIX}.auth.{REGION}.amazoncognito.com/oauth2/token"

# 3) Confidential M2M app client (client_credentials) — get-or-create by name.
GATEWAY_M2M_CLIENT_ID = None
for _c in cognito.list_user_pool_clients(UserPoolId=COGNITO_USER_POOL_ID, MaxResults=60).get("UserPoolClients", []):
    if _c.get("ClientName") == M2M_CLIENT_NAME:
        GATEWAY_M2M_CLIENT_ID = _c["ClientId"]
        break
if GATEWAY_M2M_CLIENT_ID:
    print(f"Using existing M2M client: {GATEWAY_M2M_CLIENT_ID}")
else:
    _m2m = cognito.create_user_pool_client(
        UserPoolId=COGNITO_USER_POOL_ID,
        ClientName=M2M_CLIENT_NAME,
        GenerateSecret=True,
        AllowedOAuthFlowsUserPoolClient=True,
        AllowedOAuthFlows=["client_credentials"],
        AllowedOAuthScopes=[GATEWAY_SCOPE],
        SupportedIdentityProviders=["COGNITO"],
    )
    GATEWAY_M2M_CLIENT_ID = _m2m["UserPoolClient"]["ClientId"]
    print(f"Created M2M client: {GATEWAY_M2M_CLIENT_ID}")
GATEWAY_M2M_CLIENT_SECRET = cognito.describe_user_pool_client(
    UserPoolId=COGNITO_USER_POOL_ID, ClientId=GATEWAY_M2M_CLIENT_ID
)["UserPoolClient"].get("ClientSecret", "")

print(f"\nGateway M2M Client ID:  {GATEWAY_M2M_CLIENT_ID}")
print(f"Gateway token endpoint: {GATEWAY_TOKEN_ENDPOINT}")
print(f"Gateway scope:          {GATEWAY_SCOPE}")

Created User Pool: us-west-2_nFUiYzZ7Y
Created App Client: 2siruj1ecjgstcggcm7htjb663

Cognito User Pool ID: us-west-2_nFUiYzZ7Y
Cognito Client ID:    2siruj1ecjgstcggcm7htjb663
Created resource server: anycompany-gateway (scope anycompany-gateway/invoke)
Created Cognito domain: anycompany-agentcore-nfuiyzz7y
Created M2M client: 77qsb5ks15masi1gqnsdl8op0e

Gateway M2M Client ID:  77qsb5ks15masi1gqnsdl8op0e
Gateway token endpoint: https://anycompany-agentcore-nfuiyzz7y.auth.us-west-2.amazoncognito.com/oauth2/token
Gateway scope:          anycompany-gateway/invoke


Create three test users (gold, silver, bronze) with workshop credentials. Password is stored in SSM as a SecureString immediately after.

In [8]:
# Generate a strong workshop password (or reuse the one already in SSM from a previous run).
# All three test users share this password; it is stored in SSM as a SecureString below.
import secrets, string

_ssm_for_pw = boto3.client("ssm", region_name=REGION)
_SSM_PREFIX_LOCAL = "/anycompany/agentcore"
_SSM_PASSWORD_PATH = f"{_SSM_PREFIX_LOCAL}/user_password"

def _generate_password(length: int = 16) -> str:
    # Cognito requires at least one of upper, lower, digit, symbol — seed one of each, fill, shuffle.
    upper  = secrets.choice(string.ascii_uppercase)
    lower  = secrets.choice(string.ascii_lowercase)
    digit  = secrets.choice(string.digits)
    symbol = secrets.choice("!@#$%^&*")
    rest   = [secrets.choice(string.ascii_letters + string.digits + "!@#$%^&*") for _ in range(length - 4)]
    pool   = [upper, lower, digit, symbol] + rest
    secrets.SystemRandom().shuffle(pool)
    return "".join(pool)

try:
    USER_PASSWORD = _ssm_for_pw.get_parameter(Name=_SSM_PASSWORD_PATH, WithDecryption=True)["Parameter"]["Value"]
    print(f"Reusing existing workshop password from SSM: {_SSM_PASSWORD_PATH}")
except _ssm_for_pw.exceptions.ParameterNotFound:
    USER_PASSWORD = _generate_password()
    print(f"Generated new workshop password (will be stored at: {_SSM_PASSWORD_PATH})")

TEST_USERS = [
    {"username": "gold_customer", "password": USER_PASSWORD, "email": "gold@example.com", "customer_id": "CUST-789", "tier": "gold"},
    {"username": "silver_customer", "password": USER_PASSWORD, "email": "silver@example.com", "customer_id": "CUST-456", "tier": "silver"},
    {"username": "bronze_customer", "password": USER_PASSWORD, "email": "bronze@example.com", "customer_id": "CUST-123", "tier": "bronze"},
]

for user in TEST_USERS:
    try:
        cognito.admin_create_user(
            UserPoolId=COGNITO_USER_POOL_ID,
            Username=user["username"],
            UserAttributes=[
                {"Name": "email", "Value": user["email"]},
                {"Name": "email_verified", "Value": "true"},
                {"Name": "custom:customer_id", "Value": user["customer_id"]},
                {"Name": "custom:tier", "Value": user["tier"]},
            ],
            MessageAction="SUPPRESS",
        )
        cognito.admin_set_user_password(
            UserPoolId=COGNITO_USER_POOL_ID,
            Username=user["username"],
            Password=user["password"],
            Permanent=True,
        )
        print(f"  Created: {user['username']} ({user['tier']} tier, {user['customer_id']})")
    except cognito.exceptions.UsernameExistsException:
        print(f"  Exists:  {user['username']}")

print("\nTest users ready.")

Generated new workshop password (will be stored at: /anycompany/agentcore/user_password)
  Created: gold_customer (gold tier, CUST-789)
  Created: silver_customer (silver tier, CUST-456)
  Created: bronze_customer (bronze tier, CUST-123)

Test users ready.


## Step 4: Create the AgentCore Gateway

Entry point for all tool calls. Uses MCP protocol with Cognito JWT authentication.

Create the IAM role the Gateway will assume to invoke the order and refund Lambda tools.

In [9]:
iam = boto3.client("iam")
GATEWAY_ROLE_NAME = "AnyCompanyAgentCoreGatewayRole"

gateway_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

gateway_permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "InvokeLambdaTargets",
            "Effect": "Allow",
            "Action": [
                "lambda:InvokeFunction"
            ],
            "Resource": [
                f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:anycompany_order_tools",
                f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:anycompany_refund_tools",
                f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:anycompany_registry_tools"
            ]
        }
    ]
}

try:
    role_response = iam.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(gateway_trust_policy),
        Description="Execution role for AnyCompany AgentCore Gateway"
    )
    GATEWAY_ROLE_ARN = role_response["Role"]["Arn"]
    print(f"Created role: {GATEWAY_ROLE_ARN}")

    iam.put_role_policy(
        RoleName=GATEWAY_ROLE_NAME,
        PolicyName="AnyCompanyGatewayPermissions",
        PolicyDocument=json.dumps(gateway_permissions_policy)
    )
    print("Attached inline policy.")

    # IAM role propagation takes a few seconds
    print("Waiting 10s for IAM propagation...")
    time.sleep(10)

except iam.exceptions.EntityAlreadyExistsException:
    GATEWAY_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{GATEWAY_ROLE_NAME}"
    # Update policy in case registry Lambda was added since the role was first created
    iam.put_role_policy(
        RoleName=GATEWAY_ROLE_NAME,
        PolicyName="AnyCompanyGatewayPermissions",
        PolicyDocument=json.dumps(gateway_permissions_policy)
    )
    print(f"Role already exists: {GATEWAY_ROLE_ARN} (policy updated)")

print(f"\nGateway Role ARN: {GATEWAY_ROLE_ARN}")

Created role: arn:aws:iam::359894365202:role/AnyCompanyAgentCoreGatewayRole
Attached inline policy.
Waiting 10s for IAM propagation...

Gateway Role ARN: arn:aws:iam::359894365202:role/AnyCompanyAgentCoreGatewayRole


Create the Gateway with MCP protocol and Cognito JWT auth, or look it up if it already exists.

In [10]:
print(f"Creating gateway: {GATEWAY_NAME}\n")

try:
    resp = agentcore_control.create_gateway(
        name=GATEWAY_NAME,
        description="AnyCompany agent gateway — routes requests to specialist agents via the registry.",
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        **({
            "authorizerType": "CUSTOM_JWT",
            "authorizerConfiguration": {
                "customJWTAuthorizer": {
                    "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_USER_POOL_ID}/.well-known/openid-configuration",
                    "allowedClients": [GATEWAY_M2M_CLIENT_ID],
                    "allowedScopes": [GATEWAY_SCOPE],
                }
            },
        } if COGNITO_USER_POOL_ID else {"authorizerType": "AWS_IAM"}),
    )

    GATEWAY_ID = resp["gatewayId"]
    GATEWAY_URL = resp["gatewayUrl"]
    GATEWAY_ARN = resp.get("gatewayArn", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}")
    print(f"Created gateway: {GATEWAY_NAME}")
    print(f"  ID:  {GATEWAY_ID}")
    print(f"  URL: {GATEWAY_URL}")
    print(f"  ARN: {GATEWAY_ARN}")

except agentcore_control.exceptions.ConflictException:
    print(f"Gateway '{GATEWAY_NAME}' already exists. Looking up...")
    gateways = agentcore_control.list_gateways()
    for gw in gateways.get("items", gateways.get("gateways", [])):
        if gw.get("name") == GATEWAY_NAME:
            GATEWAY_ID = gw["gatewayId"]
            # Get full details (URL, ARN) from get_gateway
            gw_detail = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID)
            GATEWAY_URL = gw_detail.get("gatewayUrl", "")
            GATEWAY_ARN = gw_detail.get("gatewayArn", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}")
            break
    else:
        raise RuntimeError(f"Gateway '{GATEWAY_NAME}' exists but could not be found in list_gateways()")
    print(f"  ID:     {GATEWAY_ID}")
    print(f"  URL:    {GATEWAY_URL}")
    print(f"  ARN:    {GATEWAY_ARN}")
    # Update authorizer to match current Cognito pool (in case pool was recreated)
    print("  Updating Gateway authorizer to current Cognito pool...")
    agentcore_control.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_USER_POOL_ID}/.well-known/openid-configuration",
                "allowedClients": [GATEWAY_M2M_CLIENT_ID],
                "allowedScopes": [GATEWAY_SCOPE],
            }
        },
    )
    print("  ✓ Authorizer updated")

# ── Normalize to the MCP invoke endpoint ────────────────────────────────
# The AgentCore API returns the gateway HOST; the MCP endpoint that accepts
# JSON-RPC tools/call is that host + "/mcp". Posting to the bare host returns
# {"Output":{"__type":"...UnknownOperationException"}}. Normalize so every
# consumer (specialist agents, the L4 test, the chatbot) uses the /mcp endpoint.
if GATEWAY_URL and not GATEWAY_URL.rstrip("/").endswith("/mcp"):
    GATEWAY_URL = GATEWAY_URL.rstrip("/") + "/mcp"
    print(f"  Normalized MCP endpoint: {GATEWAY_URL}")

Creating gateway: anycompany-agent-gateway

Created gateway: anycompany-agent-gateway
  ID:  anycompany-agent-gateway-m1jnn7lvju
  URL: https://anycompany-agent-gateway-m1jnn7lvju.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp
  ARN: arn:aws:bedrock-agentcore:us-west-2:359894365202:gateway/anycompany-agent-gateway-m1jnn7lvju


Poll until the Gateway reaches READY status.

In [11]:
# Poll until the gateway transitions to READY
import time
import time as _time

print("Waiting for gateway to be ready...")
_gw_start = _time.time()
_poll = 0

while True:
    g = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID)
    status = g.get("status")
    _elapsed = _time.time() - _gw_start
    _poll += 1

    if status == "READY":
        print(f"Gateway is READY after {_elapsed:.1f}s ({_poll} poll(s))")
        break

    if status in {"FAILED", "UPDATE_UNSUCCESSFUL", "DELETING"}:
        reasons = g.get("statusReasons", [])
        raise RuntimeError(f"Gateway entered terminal state {status}: {reasons}")

    print(f"  [{_elapsed:5.1f}s] Status: {status} - waiting... (poll #{_poll})")
    time.sleep(3)

print(f"\nGateway ID:  {GATEWAY_ID}")
print(f"Gateway URL: {GATEWAY_URL}")
print(f"Gateway ARN: {GATEWAY_ARN}")

Waiting for gateway to be ready...
  [  0.1s] Status: CREATING - waiting... (poll #1)
Gateway is READY after 3.1s (2 poll(s))

Gateway ID:  anycompany-agent-gateway-m1jnn7lvju
Gateway URL: https://anycompany-agent-gateway-m1jnn7lvju.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp
Gateway ARN: arn:aws:bedrock-agentcore:us-west-2:359894365202:gateway/anycompany-agent-gateway-m1jnn7lvju


## Step 5: Registry Discovery Tool (Gateway Target)

The orchestrator Harness is fully managed — it cannot run arbitrary code to call the
Registry directly. Instead of teaching it the registry API and A2A protocol in its system
prompt, we expose registry **search** and **invoke** as MCP tools behind a second, IAM-auth
Gateway (`anycompany-harness-gateway`).

A small Lambda (`anycompany_registry_tools`) wraps `search_discoverable_registry_records`
and `invoke_agent_runtime` from the `agent-registry` API (bundled with the latest boto3,
since the default Lambda runtime does not yet include that client). The Harness then calls
`registry-tools___search_agents` and `registry-tools___invoke_agent` like any other tool —
discovery and delegation become reliable tool calls rather than prompt-driven improvisation.

This runs after the tool Gateway above and before the SSM step, so `harness_gateway_arn`
is saved for the orchestrator (Notebook 4) to consume.

In [12]:
# Create IAM-auth Gateway for the Harness (separate from the Cognito JWT gateway used by sub-agents)
IAM_GATEWAY_NAME = "anycompany-harness-gateway"

try:
    resp = agentcore_control.create_gateway(
        name=IAM_GATEWAY_NAME,
        description="IAM-auth gateway for the Harness to access registry discovery tools",
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        authorizerType="AWS_IAM",
    )
    IAM_GATEWAY_ID = resp["gatewayId"]
    IAM_GATEWAY_ARN = resp.get("gatewayArn", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{IAM_GATEWAY_ID}")
    print(f"Created IAM gateway: {IAM_GATEWAY_NAME}")
except agentcore_control.exceptions.ConflictException:
    print(f"IAM gateway '{IAM_GATEWAY_NAME}' already exists. Looking up...")
    gateways = agentcore_control.list_gateways()
    for gw in gateways.get("items", gateways.get("gateways", [])):
        if gw.get("name") == IAM_GATEWAY_NAME:
            IAM_GATEWAY_ID = gw["gatewayId"]
            gw_detail = agentcore_control.get_gateway(gatewayIdentifier=IAM_GATEWAY_ID)
            IAM_GATEWAY_ARN = gw_detail.get("gatewayArn", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{IAM_GATEWAY_ID}")
            break

print(f"  ID:  {IAM_GATEWAY_ID}")
print(f"  ARN: {IAM_GATEWAY_ARN}")

# Wait for ready
for _ in range(20):
    gw = agentcore_control.get_gateway(gatewayIdentifier=IAM_GATEWAY_ID)
    if gw.get("status") == "READY":
        print("  Status: READY")
        break
    time.sleep(3)

import io, zipfile

# Lambda code: wraps Agent Registry search + Agent Runtime invocation
REGISTRY_LAMBDA_CODE = '''import json, os, boto3, logging, urllib.parse, uuid

logger = logging.getLogger()
logger.setLevel(logging.INFO)

REGION = os.environ.get("AGENT_REGION", "us-east-1")
REGISTRY_ID = os.environ.get("REGISTRY_ID", "")

agent_registry_dp = boto3.client("agent-registry", region_name=REGION)
agentcore_runtime = boto3.client("bedrock-agentcore", region_name=REGION)


def _extract_arn_from_url(url):
    if "/runtimes/" in url:
        encoded_arn = url.split("/runtimes/")[1].split("/invocations")[0]
        return urllib.parse.unquote(encoded_arn)
    return ""


def lambda_handler(event, context):
    logger.info(f"Registry tool invoked: {json.dumps(event)}")
    action = event.get("action", "search")

    if action == "search":
        query = event.get("search_query", event.get("query", ""))
        if not query:
            return {"status": "error", "message": "search_query is required"}
        try:
            resp = agent_registry_dp.search_discoverable_registry_records(
                registryIds=[REGISTRY_ID], searchQuery=query,
            )
            results = []
            for r in resp.get("registryRecords", []):
                agent_arn = ""
                descriptors = r.get("descriptors", {})
                if isinstance(descriptors, dict):
                    a2a_card = descriptors.get("a2aAgentCard", {})
                    if isinstance(a2a_card, dict):
                        data = a2a_card.get("data", "")
                        if data:
                            try:
                                parsed_card = json.loads(data)
                                url = parsed_card.get("url", "")
                                if url:
                                    agent_arn = _extract_arn_from_url(url)
                            except json.JSONDecodeError:
                                pass
                results.append({
                    "name": r.get("name", ""),
                    "description": r.get("description", ""),
                    "agentRuntimeArn": agent_arn,
                    "status": r.get("status", ""),
                })
            return {"status": "success", "agents": results, "count": len(results)}
        except Exception as e:
            logger.error(f"Search failed: {e}")
            return {"status": "error", "message": str(e)}

    elif action == "invoke":
        agent_arn = event.get("agent_runtime_arn", "")
        message = event.get("message", "")
        if not agent_arn or not message:
            return {"status": "error", "message": "agent_runtime_arn and message are required"}
        try:
            a2a_payload = {
                "jsonrpc": "2.0",
                "id": str(uuid.uuid4()),
                "method": "message/send",
                "params": {
                    "message": {
                        "messageId": str(uuid.uuid4()),
                        "role": "user",
                        "parts": [{"kind": "text", "text": message}],
                    }
                },
            }
            resp = agentcore_runtime.invoke_agent_runtime(
                agentRuntimeArn=agent_arn,
                qualifier="DEFAULT",
                payload=json.dumps(a2a_payload),
                contentType="application/json",
            )
            body = resp.get("response", "")
            if hasattr(body, "read"):
                body = body.read().decode("utf-8")
            try:
                parsed = json.loads(body)
                artifacts = parsed.get("result", {}).get("artifacts", [])
                text_parts = []
                for artifact in artifacts:
                    for part in artifact.get("parts", []):
                        if part.get("kind") == "text":
                            text_parts.append(part.get("text", ""))
                if text_parts:
                    return {"status": "success", "response": " ".join(text_parts)}
            except json.JSONDecodeError:
                pass
            return {"status": "success", "response": body[:2000] if body else "Agent returned empty response"}
        except Exception as e:
            logger.error(f"Invoke failed: {e}")
            return {"status": "error", "message": str(e)}

    return {"status": "error", "message": f"Unknown action: {action}. Use search or invoke."}
'''

# IAM role for the Registry Lambda
REGISTRY_LAMBDA_ROLE_NAME = "AnyCompanyRegistryToolsLambdaRole"
registry_lambda_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})
registry_lambda_perms = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:/aws/lambda/anycompany_registry_tools"},
        {"Effect": "Allow", "Action": ["logs:CreateLogStream", "logs:PutLogEvents"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:/aws/lambda/anycompany_registry_tools:*"},
        {"Effect": "Allow", "Action": ["agent-registry:SearchDiscoverableRegistryRecords", "agent-registry:ListDiscoverableRegistryRecords", "agent-registry:GetDiscoverableRegistryRecord"], "Resource": f"arn:aws:agent-registry:{REGION}:{ACCOUNT_ID}:*"},
        {"Effect": "Allow", "Action": ["bedrock-agentcore:InvokeAgentRuntime"], "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:runtime/*"},
    ]
})

iam = boto3.client("iam")
try:
    r = iam.create_role(RoleName=REGISTRY_LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=registry_lambda_trust)
    REGISTRY_LAMBDA_ROLE_ARN = r["Role"]["Arn"]
    iam.put_role_policy(RoleName=REGISTRY_LAMBDA_ROLE_NAME, PolicyName="RegistryToolsPerms", PolicyDocument=registry_lambda_perms)
    print(f"Created role: {REGISTRY_LAMBDA_ROLE_ARN}")
    time.sleep(10)
except iam.exceptions.EntityAlreadyExistsException:
    REGISTRY_LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{REGISTRY_LAMBDA_ROLE_NAME}"
    iam.put_role_policy(RoleName=REGISTRY_LAMBDA_ROLE_NAME, PolicyName="RegistryToolsPerms", PolicyDocument=registry_lambda_perms)
    print(f"Exists: {REGISTRY_LAMBDA_ROLE_ARN}")

# Package and deploy Lambda (with bundled boto3 for agent-registry support)
import subprocess, tempfile, shutil
buf = io.BytesIO()
with tempfile.TemporaryDirectory() as tmpdir:
    # Install boto3 into the package directory
    subprocess.run(
        ['pip', 'install', 'boto3', '-t', tmpdir, '-q'],
        check=True
    )
    # Write the Lambda code
    with open(f'{tmpdir}/lambda_function.py', 'w') as f:
        f.write(REGISTRY_LAMBDA_CODE)
    # Zip everything
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(tmpdir):
            for file in files:
                filepath = os.path.join(root, file)
                arcname = os.path.relpath(filepath, tmpdir)
                zf.write(filepath, arcname)
zip_bytes = buf.getvalue()
print(f"Lambda package size: {len(zip_bytes) / 1024 / 1024:.1f} MB")

REGISTRY_LAMBDA_NAME = "anycompany_registry_tools"
lambda_client = boto3.client("lambda", region_name=REGION)

try:
    resp = lambda_client.create_function(
        FunctionName=REGISTRY_LAMBDA_NAME,
        Runtime="python3.12",
        Role=REGISTRY_LAMBDA_ROLE_ARN,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=60,
        MemorySize=256,
        Environment={"Variables": {"REGISTRY_ID": REGISTRY_ID, "AGENT_REGION": REGION}},
    )
    REGISTRY_LAMBDA_ARN = resp["FunctionArn"]
    print(f"Created: {REGISTRY_LAMBDA_ARN}")
except lambda_client.exceptions.ResourceConflictException:
    REGISTRY_LAMBDA_ARN = lambda_client.get_function(FunctionName=REGISTRY_LAMBDA_NAME)["Configuration"]["FunctionArn"]
    lambda_client.update_function_code(FunctionName=REGISTRY_LAMBDA_NAME, ZipFile=zip_bytes)
    lambda_client.get_waiter('function_updated_v2').wait(FunctionName=REGISTRY_LAMBDA_NAME)
    lambda_client.update_function_configuration(
        FunctionName=REGISTRY_LAMBDA_NAME,
        Environment={"Variables": {"REGISTRY_ID": REGISTRY_ID, "AGENT_REGION": REGION}},
    )
    print(f"Updated: {REGISTRY_LAMBDA_ARN}")

# Allow Gateway to invoke
try:
    lambda_client.add_permission(
        FunctionName=REGISTRY_LAMBDA_NAME,
        StatementId="AllowGatewayInvoke",
        Action="lambda:InvokeFunction",
        Principal="bedrock-agentcore.amazonaws.com",
    )
    print("Added Gateway invoke permission.")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists.")

print(f"\nRegistry Lambda ready: {REGISTRY_LAMBDA_ARN}")

Created IAM gateway: anycompany-harness-gateway
  ID:  anycompany-harness-gateway-5qen0ajy4t
  ARN: arn:aws:bedrock-agentcore:us-west-2:359894365202:gateway/anycompany-harness-gateway-5qen0ajy4t
  Status: READY
Created role: arn:aws:iam::359894365202:role/AnyCompanyRegistryToolsLambdaRole


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
sagemaker-studio 1.1.32 requires pymongo-auth-aws>=1.3.0, which is not installed.
sagemaker-studio 1.1.32 requires pymongosql>=0.7.2, which is not installed.
aiobotocore 3.9.0 requires aiohttp<4.0.0,>=3.14.0, but you have aiohttp 3.13.5 which is incompatible.
aiobotocore 3.9.0 requires botocore<1.43.57,>=1.43.3, but you have botocore 1.43.109 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.7.0 which is incompatible.
sagemaker-core 2.21.0 requires importlib-metadata<7.0,>=1.4.0, but you have importlib-metadata 8.5.0 which is incompatible.
sagemaker-studio-analytics-

Lambda package size: 17.1 MB
Created: arn:aws:lambda:us-west-2:359894365202:function:anycompany_registry_tools
Added Gateway invoke permission.

Registry Lambda ready: arn:aws:lambda:us-west-2:359894365202:function:anycompany_registry_tools


In [13]:
# Create Gateway Target for Registry tools
REGISTRY_TOOL_SCHEMAS = [
    {
        "name": "search_agents",
        "description": "Search the Agent Registry for specialist agents by description. Returns agent names, descriptions, and runtime ARNs that can be used with invoke_agent.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "action": {"type": "string", "description": "Must be the string: search"},
                "search_query": {"type": "string", "description": "Natural language description of the agent you need (e.g. 'order management' or 'refund processing')"}
            },
            "required": ["action", "search_query"]
        }
    },
    {
        "name": "invoke_agent",
        "description": "Invoke a specialist agent discovered from the Registry. Pass the agentRuntimeArn from search_agents results and a natural language message.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "action": {"type": "string", "description": "Must be the string: invoke"},
                "agent_runtime_arn": {"type": "string", "description": "The agentRuntimeArn from search_agents results"},
                "message": {"type": "string", "description": "The message/query to send to the specialist agent"}
            },
            "required": ["action", "agent_runtime_arn", "message"]
        }
    }
]

REGISTRY_TARGET_NAME = "registry-tools"

# Wait for Lambda to be active
print("Waiting for Lambda to be active...")
for _ in range(20):
    fn = lambda_client.get_function_configuration(FunctionName=REGISTRY_LAMBDA_NAME)
    if fn.get("State") == "Active" and fn.get("LastUpdateStatus", "Successful") in ("Successful", None):
        break
    time.sleep(5)

try:
    target_resp = agentcore_control.create_gateway_target(
        gatewayIdentifier=IAM_GATEWAY_ID,
        name=REGISTRY_TARGET_NAME,
        description="Agent Registry discovery and invocation tools — search for agents and invoke them via A2A",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": REGISTRY_LAMBDA_ARN,
                    "toolSchema": {"inlinePayload": REGISTRY_TOOL_SCHEMAS}
                }
            }
        },
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    REGISTRY_TARGET_ID = target_resp["targetId"]
    print(f"Created Gateway Target: {REGISTRY_TARGET_ID}")
except agentcore_control.exceptions.ConflictException:
    targets = agentcore_control.list_gateway_targets(gatewayIdentifier=IAM_GATEWAY_ID)
    for t in targets.get("targets", targets.get("items", [])):
        if t.get("name") == REGISTRY_TARGET_NAME:
            REGISTRY_TARGET_ID = t["targetId"]
            print(f"Already exists: {REGISTRY_TARGET_ID} — updating tool schemas...")
            agentcore_control.update_gateway_target(
                gatewayIdentifier=IAM_GATEWAY_ID,
                targetId=REGISTRY_TARGET_ID,
                name=REGISTRY_TARGET_NAME,
                description="Agent Registry discovery and invocation tools — search for agents and invoke them via A2A",
                targetConfiguration={
                    "mcp": {
                        "lambda": {
                            "lambdaArn": REGISTRY_LAMBDA_ARN,
                            "toolSchema": {"inlinePayload": REGISTRY_TOOL_SCHEMAS}
                        }
                    }
                },
                credentialProviderConfigurations=[
                    {"credentialProviderType": "GATEWAY_IAM_ROLE"}
                ],
            )
            print(f"Updated target: {REGISTRY_TARGET_ID}")
            break

# Wait for target to be ready
print("Waiting for target...")
for _ in range(20):
    t = agentcore_control.get_gateway_target(gatewayIdentifier=IAM_GATEWAY_ID, targetId=REGISTRY_TARGET_ID)
    status = t["status"]
    if status == "READY":
        print(f"Gateway Target READY")
        break
    print(f"  {status}...")
    time.sleep(5)

print(f"\nRegistry tools available via Gateway: search_agents, invoke_agent")

Waiting for Lambda to be active...
Created Gateway Target: WM6Z4BHSYL
Waiting for target...
  CREATING...
Gateway Target READY

Registry tools available via Gateway: search_agents, invoke_agent


## Step 6: Save Configuration to SSM Parameter Store

Publish all resource IDs to SSM so Notebooks 2–4, L4, L5, and L6 can read them without re-creating resources.

In [14]:
ssm = boto3.client("ssm", region_name=REGION)

SSM_PREFIX = "/anycompany/agentcore"

parameters = {
    f"{SSM_PREFIX}/region": REGION,
    f"{SSM_PREFIX}/account_id": ACCOUNT_ID,
    f"{SSM_PREFIX}/registry_id": REGISTRY_ID,
    f"{SSM_PREFIX}/registry_arn": REGISTRY_ARN,
    f"{SSM_PREFIX}/registry_name": REGISTRY_NAME,
    f"{SSM_PREFIX}/gateway_id": GATEWAY_ID,
    f"{SSM_PREFIX}/gateway_arn": GATEWAY_ARN,
    f"{SSM_PREFIX}/gateway_url": GATEWAY_URL,
    f"{SSM_PREFIX}/gateway_name": GATEWAY_NAME,
    f"{SSM_PREFIX}/gateway_role_arn": GATEWAY_ROLE_ARN,
    f"{SSM_PREFIX}/harness_gateway_arn": IAM_GATEWAY_ARN,
    f"{SSM_PREFIX}/cognito_user_pool_id": COGNITO_USER_POOL_ID,
    f"{SSM_PREFIX}/cognito_client_id": COGNITO_CLIENT_ID,
    f"{SSM_PREFIX}/gateway_m2m_client_id": GATEWAY_M2M_CLIENT_ID,
    f"{SSM_PREFIX}/gateway_token_endpoint": GATEWAY_TOKEN_ENDPOINT,
    f"{SSM_PREFIX}/gateway_scope": GATEWAY_SCOPE,
}

# Sensitive values stored as SecureString (encrypted at rest with KMS).
# Consumers must call get_parameter(..., WithDecryption=True) to read them.
secure_parameters = {
    f"{SSM_PREFIX}/user_password": USER_PASSWORD,
    f"{SSM_PREFIX}/gateway_m2m_client_secret": GATEWAY_M2M_CLIENT_SECRET,
}

print(f"Saving {len(parameters)} String + {len(secure_parameters)} SecureString parameters to SSM under '{SSM_PREFIX}/'\n")

for name, value in parameters.items():
    ssm.put_parameter(
        Name=name,
        Value=value,
        Type="String",
        Overwrite=True,
    )
    print(f"  ✓ {name} = {value}")

for name, value in secure_parameters.items():
    ssm.put_parameter(
        Name=name,
        Value=value,
        Type="SecureString",
        Overwrite=True,
    )
    print(f"  ✓ {name} = *** (SecureString)")

print(f"\nAll parameters saved to SSM under prefix: {SSM_PREFIX}/")

Saving 16 String + 2 SecureString parameters to SSM under '/anycompany/agentcore/'

  ✓ /anycompany/agentcore/region = us-west-2
  ✓ /anycompany/agentcore/account_id = 359894365202
  ✓ /anycompany/agentcore/registry_id = XCtvGlP9E1u5ZoYr
  ✓ /anycompany/agentcore/registry_arn = arn:aws:agent-registry:us-west-2:359894365202:registry/XCtvGlP9E1u5ZoYr
  ✓ /anycompany/agentcore/registry_name = anycompany-agent-registry
  ✓ /anycompany/agentcore/gateway_id = anycompany-agent-gateway-m1jnn7lvju
  ✓ /anycompany/agentcore/gateway_arn = arn:aws:bedrock-agentcore:us-west-2:359894365202:gateway/anycompany-agent-gateway-m1jnn7lvju
  ✓ /anycompany/agentcore/gateway_url = https://anycompany-agent-gateway-m1jnn7lvju.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp
  ✓ /anycompany/agentcore/gateway_name = anycompany-agent-gateway
  ✓ /anycompany/agentcore/gateway_role_arn = arn:aws:iam::359894365202:role/AnyCompanyAgentCoreGatewayRole
  ✓ /anycompany/agentcore/harness_gateway_arn = arn:aws:bedroc

## Step 7: Store Datadog Credentials in SSM

The Datadog track deploys agents with a trace-agent sidecar that ships APM spans to Datadog.
The agents read `DD_API_KEY` and `DD_SITE` from SSM at deploy time (Notebooks 2 and 3).
This step validates your Datadog credentials and stores them for downstream use.

Enter your Datadog API key and Application key, validate them against the Datadog API, and store in SSM as SecureString.

**Application key scopes matter.** Scope the Application key for everything the later notebooks do with it. This is the full set the workshop exercises:

| Scope | Needed by |
|-------|-----------|
| `ai_guard_evaluate` | L4 — AI Guard evaluations |
| `ai_guard_view` | L4 — read AI Guard service policy / configuration |
| `ai_guard_write` | L4 — set the AI Guard service policy |
| `apm_read` | L6 — reads traces/spans via the Datadog API (`list_spans`) |
| `metrics_read` | L6 nb2 — reads metrics |
| `timeseries_query` | L6 nb2 — queries metric timeseries for the dashboard |
| `monitors_write` | L6 nb2 — creates the Datadog Monitors |
| `dashboards_write` | L6 nb2 — creates the operational Dashboard |
| `slos_write` | L6 nb5 — creates the SLO (and deletes it in cleanup) |
| `llm_observability_read` | L6 nb4, L7 nb2 — reads Agent Observability datasets and experiments |
| `llm_observability_write` | L6 nb4, L7 nb2 — creates evaluations, datasets, and experiments |

A key missing a scope returns `403 Failed permission authorization checks`. Two things that are easy to miss:

- **Scopes are capped by the owning user's role.** A scoped key grants only the permissions its user already holds, so if your role is read-only, adding a `*_write` scope to the key does nothing — the role needs the permission too.
- **Agent Observability needs both `llm_observability_read` and `llm_observability_write`.** Either one alone returns 403 when creating a dataset or experiment (L7 nb2).

Where a scope is missing, the `*_write` cells in L6 nb2 and nb5 fall back to printing import-ready JSON so you can create the monitor, dashboard, or SLO in the UI. L7 nb2 has no such fallback — creating an experiment is inherently a write, so `llm_observability_read` + `llm_observability_write` are required to run it.

In [15]:
# Store Datadog credentials in SSM for agent telemetry (L3 deploy + L6 observability)
from datadog_api_client import Configuration, ApiClient
from datadog_api_client.v1.api.authentication_api import AuthenticationApi

# ============================================================
# ENTER YOUR DATADOG CREDENTIALS BELOW
# Get API key from: https://app.datadoghq.com/organization-settings/api-keys
# Get App key from: https://app.datadoghq.com/organization-settings/application-keys
#   The Application key must carry these scopes (full set the workshop uses):
#     - `ai_guard_evaluate`       -> L4 AI Guard evaluations
#     - `ai_guard_view`           -> L4 read AI Guard service policy / config
#     - `ai_guard_write`          -> L4 set the AI Guard service policy
#     - `apm_read`                -> L6 reads traces/spans (list_spans)
#     - `metrics_read`            -> L6 nb2 reads metrics
#     - `timeseries_query`        -> L6 nb2 queries metric timeseries for the dashboard
#     - `monitors_write`          -> L6 nb2 creates the Datadog Monitors
#     - `dashboards_write`        -> L6 nb2 creates the operational Dashboard
#     - `slos_write`              -> L6 nb5 creates the SLO
#     - `llm_observability_read`  -> L6 nb4, L7 nb2 read datasets/experiments
#     - `llm_observability_write` -> L6 nb4, L7 nb2 create evaluations/datasets/experiments
#   A key missing a scope returns 403 "Failed permission authorization checks".
#   A scoped key is also capped by its owning user's role, so the role must hold the
#   permission too. Agent Observability (L7 nb2) needs BOTH llm_observability_read and
#   llm_observability_write - either alone returns 403.
# ============================================================
DD_API_KEY = os.environ.get("DD_API_KEY", "")  # Your Datadog API key
DD_APP_KEY = os.environ.get("DD_APP_KEY", "")  # Your Datadog Application key
DD_SITE = os.environ.get("DD_SITE", "datadoghq.com")  # See the site list below if validation fails

# If env vars are not set, prompt the user
if not DD_API_KEY:
    DD_API_KEY = input("Enter your Datadog API Key: ").strip()
if not DD_APP_KEY:
    DD_APP_KEY = input("Enter your Datadog Application Key: ").strip()
if not DD_API_KEY or not DD_APP_KEY:
    raise RuntimeError("Both DD_API_KEY and DD_APP_KEY are required to proceed.")

# Validate credentials against Datadog API
dd_config = Configuration()
dd_config.api_key["apiKeyAuth"] = DD_API_KEY
dd_config.api_key["appKeyAuth"] = DD_APP_KEY
dd_config.server_variables["site"] = DD_SITE

try:
    with ApiClient(dd_config) as client:
        resp = AuthenticationApi(client).validate()
        if not resp.get("valid"):
            raise RuntimeError(f"Datadog reported the key as invalid: {resp}")
except Exception as e:
    raise RuntimeError(
        f"Datadog API validation failed for site '{DD_SITE}'.\n"
        f"\n"
        f"Your org may be on a different Datadog site. Check the URL when logged in:\n"
        f"  app.datadoghq.com      -> DD_SITE = 'datadoghq.com'\n"
        f"  us3.datadoghq.com      -> DD_SITE = 'us3.datadoghq.com'\n"
        f"  us5.datadoghq.com      -> DD_SITE = 'us5.datadoghq.com'\n"
        f"  app.datadoghq.eu       -> DD_SITE = 'datadoghq.eu'\n"
        f"  ap1.datadoghq.com      -> DD_SITE = 'ap1.datadoghq.com'\n"
        f"  ap2.datadoghq.com      -> DD_SITE = 'ap2.datadoghq.com'\n"
        f"\n"
        f"Set it and re-run this cell:\n"
        f"    os.environ['DD_SITE'] = '<your-site>'\n"
        f"\n"
        f"If the site is correct, verify the API key and Application key are from this org.\n"
        f"Original error: {e}"
    ) from e

print(f"Datadog API: validated (site: {DD_SITE})")

# Store in SSM (keys as SecureString, site as String)
dd_params = [
    (f"{SSM_PREFIX}/dd_api_key", DD_API_KEY, "SecureString"),
    (f"{SSM_PREFIX}/dd_app_key", DD_APP_KEY, "SecureString"),
    (f"{SSM_PREFIX}/dd_site", DD_SITE, "String"),
]

for name, value, ptype in dd_params:
    ssm.put_parameter(Name=name, Value=value, Type=ptype, Overwrite=True)
    print(f"  ✓ {name} (Type={ptype})")

print(f"\nDatadog credentials stored. Notebooks 2-3 will read these at deploy time.")


Enter your Datadog API Key:  KEY HERE
Enter your Datadog Application Key:  KEY HERE


Datadog API: validated (site: datadoghq.com)
  ✓ /anycompany/agentcore/dd_api_key (Type=SecureString)
  ✓ /anycompany/agentcore/dd_app_key (Type=SecureString)
  ✓ /anycompany/agentcore/dd_site (Type=String)

Datadog credentials stored. Notebooks 2-3 will read these at deploy time.


## Summary

| Resource | Name | Purpose |
|----------|------|---------|
| AgentCore Registry | `anycompany-agent-registry` | Specialist agents publish themselves here at deploy time |
| AgentCore Gateway | `anycompany-agent-gateway` | Routes all tool calls to the correct Lambda target |
| Cognito User Pool | `AnyCompanyAgentPool` | Authenticates chatbot users via JWT |
| SSM parameters | `/anycompany/agentcore/...` | Shared config consumed by Notebooks 2–5, L4, and L5 |
| Datadog credentials | `/anycompany/agentcore/dd_*` | API key, App key, and site for agent telemetry |

### What's Next

| Notebook | What It Does |
|----------|--------------|
| `2_order_agent.ipynb` | Deploys the Order Agent and registers it in the Registry |
| `3_refund_agent.ipynb` | Deploys the Refund Agent and registers it in the Registry |
| `4_orchestrator_agent.ipynb` | Creates the Harness that discovers and delegates to both agents |
| `l5-application/1_chatbot_ui.ipynb` | Wires the system into a Streamlit chatbot UI |

---
## Cleanup

Run the following cell to delete all resources created in this notebook and avoid ongoing charges.

**Resources deleted:**
- AgentCore Registry and Gateway
- Cognito User Pool
- Gateway IAM role
- SSM parameters (including Datadog credentials)

Uncomment and run to delete all resources created in this notebook.

In [16]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Account-level changes from Step 1 (X-Ray Transaction Search resource policy,
##       trace destination, indexing rule, Application Signals discovery) are
##       intentionally NOT reverted — they are idempotent and useful to keep enabled.

# import boto3, os, time
# from botocore.exceptions import ClientError
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
# REGISTRY_NAME = "anycompany-agent-registry"
# GATEWAY_NAME = "anycompany-agent-gateway"
# POOL_NAME = "AnyCompanyAgentPool"
# GATEWAY_ROLE_NAME = "AnyCompanyAgentCoreGatewayRole"
# HARNESS_GATEWAY_NAME = "anycompany-harness-gateway"
# REGISTRY_LAMBDA_NAME = "anycompany_registry_tools"
# REGISTRY_LAMBDA_ROLE_NAME = "AnyCompanyRegistryToolsLambdaRole"
#
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# agent_registry = boto3.client("agent-registry-control", region_name=REGION)
# cognito = boto3.client("cognito-idp", region_name=REGION)
# iam = boto3.client("iam")
# ssm = boto3.client("ssm", region_name=REGION)
# lambda_client = boto3.client("lambda", region_name=REGION)
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# # Resolve IDs from SSM, falling back to lookup-by-name (idempotent on partial re-runs).
# GATEWAY_ID = _ssm_get("gateway_id")
# if not GATEWAY_ID:
#     for gw in agentcore_control.list_gateways().get("items", agentcore_control.list_gateways().get("gateways", [])):
#         if gw.get("name") == GATEWAY_NAME:
#             GATEWAY_ID = gw["gatewayId"]
#             break
#
# HARNESS_GATEWAY_ID = None
# _harness_arn = _ssm_get("harness_gateway_arn")
# if _harness_arn:
#     HARNESS_GATEWAY_ID = _harness_arn.split("/")[-1]
# else:
#     for gw in agentcore_control.list_gateways().get("items", agentcore_control.list_gateways().get("gateways", [])):
#         if gw.get("name") == HARNESS_GATEWAY_NAME:
#             HARNESS_GATEWAY_ID = gw["gatewayId"]
#             break
#
# REGISTRY_ID = _ssm_get("registry_id")
# if not REGISTRY_ID:
#     for reg in agent_registry.list_registries().get("registries", []):
#         if reg.get("name") == REGISTRY_NAME:
#             REGISTRY_ID = reg["registryArn"].split("/")[-1]
#             break
#
# COGNITO_USER_POOL_ID = _ssm_get("cognito_user_pool_id")
# if not COGNITO_USER_POOL_ID:
#     pools = cognito.list_user_pools(MaxResults=60).get("UserPools", [])
#     for p in pools:
#         if p.get("Name") == POOL_NAME:
#             COGNITO_USER_POOL_ID = p["Id"]
#             break
#
# # ── Phase 1: Drain registry records (downstream notebooks register agents here) ──
# if REGISTRY_ID:
#     try:
#         records = agent_registry.list_registry_records(registryId=REGISTRY_ID).get("registryRecords", [])
#         for rec in records:
#             try:
#                 agent_registry.delete_registry_record(
#                     registryId=REGISTRY_ID,
#                     recordId=rec["recordId"],
#                 )
#                 print(f"Deleted registry record: {rec.get('name', rec['recordId'])}")
#             except Exception as e:
#                 print(f"Registry record cleanup ({rec.get('name')}): {e}")
#         if records:
#             time.sleep(5)  # let async record deletes propagate before deleting registry
#     except Exception as e:
#         print(f"List registry records: {e}")
#
# # ── Phase 2: Delete Gateway and poll until gone ─────────────────────────────────
# if GATEWAY_ID:
#     try:
#         agentcore_control.delete_gateway(gatewayIdentifier=GATEWAY_ID)
#         print(f"Initiated delete of Gateway: {GATEWAY_ID}")
#         deadline = time.time() + 300  # 5 min
#         while time.time() < deadline:
#             try:
#                 status = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID).get("status")
#                 print(f"  Gateway status: {status}")
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print(f"Gateway {GATEWAY_ID} fully deleted.")
#                 break
#             time.sleep(10)
#         else:
#             print("Timed out waiting for Gateway to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Gateway {GATEWAY_ID} already deleted.")
#     except Exception as e:
#         print(f"Gateway cleanup: {e}")
#
# # ── Phase 2b: Delete Harness Gateway (registry-discovery) and poll until gone ──
# if HARNESS_GATEWAY_ID:
#     try:
#         agentcore_control.delete_gateway(gatewayIdentifier=HARNESS_GATEWAY_ID)
#         print(f"Initiated delete of Harness Gateway: {HARNESS_GATEWAY_ID}")
#         deadline = time.time() + 300
#         while time.time() < deadline:
#             try:
#                 status = agentcore_control.get_gateway(gatewayIdentifier=HARNESS_GATEWAY_ID).get("status")
#                 print(f"  Harness Gateway status: {status}")
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print(f"Harness Gateway {HARNESS_GATEWAY_ID} fully deleted.")
#                 break
#             time.sleep(10)
#         else:
#             print("Timed out waiting for Harness Gateway to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Harness Gateway {HARNESS_GATEWAY_ID} already deleted.")
#     except Exception as e:
#         print(f"Harness Gateway cleanup: {e}")
#
# # ── Phase 3: Delete Registry (records were drained in Phase 1) ──────────────────
# if REGISTRY_ID:
#     try:
#         agent_registry.delete_registry(registryId=REGISTRY_ID)
#         print(f"Initiated delete of Registry: {REGISTRY_ID}")
#         deadline = time.time() + 300
#         while time.time() < deadline:
#             try:
#                 status = agent_registry.get_registry(registryId=REGISTRY_ID).get("status")
#                 print(f"  Registry status: {status}")
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print(f"Registry {REGISTRY_ID} fully deleted.")
#                 break
#             time.sleep(10)
#         else:
#             print("Timed out waiting for Registry to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Registry {REGISTRY_ID} already deleted.")
#     except Exception as e:
#         print(f"Registry cleanup: {e}")
#
# # ── Phase 4: Delete Cognito User Pool (cascades to users and app client) ────────
# if COGNITO_USER_POOL_ID:
#     try:
#         _dom = cognito.describe_user_pool(UserPoolId=COGNITO_USER_POOL_ID)["UserPool"].get("Domain")
#         if _dom:
#             cognito.delete_user_pool_domain(Domain=_dom, UserPoolId=COGNITO_USER_POOL_ID)
#             print(f"Deleted Cognito domain: {_dom}")
#         cognito.delete_user_pool(UserPoolId=COGNITO_USER_POOL_ID)
#         print(f"Deleted Cognito User Pool: {COGNITO_USER_POOL_ID}")
#     except cognito.exceptions.ResourceNotFoundException:
#         print(f"Cognito User Pool {COGNITO_USER_POOL_ID} already deleted.")
#     except Exception as e:
#         print(f"Cognito cleanup: {e}")
#
# # ── Phase 5: Delete Gateway IAM role (after Gateway is gone — Phase 2) ──────────
# try:
#     for p in iam.list_role_policies(RoleName=GATEWAY_ROLE_NAME).get("PolicyNames", []):
#         iam.delete_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName=p)
#     for ap in iam.list_attached_role_policies(RoleName=GATEWAY_ROLE_NAME).get("AttachedPolicies", []):
#         iam.detach_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyArn=ap["PolicyArn"])
#     iam.delete_role(RoleName=GATEWAY_ROLE_NAME)
#     print(f"Deleted IAM role: {GATEWAY_ROLE_NAME}")
# except iam.exceptions.NoSuchEntityException:
#     print(f"IAM role {GATEWAY_ROLE_NAME} already deleted.")
# except Exception as e:
#     print(f"IAM cleanup: {e}")
#
# # ── Phase 5b: Delete registry-discovery Lambda and its execution role ──
# try:
#     lambda_client.delete_function(FunctionName=REGISTRY_LAMBDA_NAME)
#     print(f"Deleted Lambda: {REGISTRY_LAMBDA_NAME}")
# except lambda_client.exceptions.ResourceNotFoundException:
#     print(f"Lambda {REGISTRY_LAMBDA_NAME} already deleted.")
# except Exception as e:
#     print(f"Registry Lambda cleanup: {e}")
# try:
#     for p in iam.list_role_policies(RoleName=REGISTRY_LAMBDA_ROLE_NAME).get("PolicyNames", []):
#         iam.delete_role_policy(RoleName=REGISTRY_LAMBDA_ROLE_NAME, PolicyName=p)
#     iam.delete_role(RoleName=REGISTRY_LAMBDA_ROLE_NAME)
#     print(f"Deleted IAM role: {REGISTRY_LAMBDA_ROLE_NAME}")
# except iam.exceptions.NoSuchEntityException:
#     print(f"IAM role {REGISTRY_LAMBDA_ROLE_NAME} already deleted.")
# except Exception as e:
#     print(f"Registry Lambda role cleanup: {e}")
#
# # ── Phase 6: Delete SSM parameters (last) ───────────────────────────────────────
# ssm_params = [
#     "region", "account_id",
#     "registry_id", "registry_arn", "registry_name",
#     "gateway_id", "gateway_arn", "gateway_url", "gateway_name", "gateway_role_arn",
#     "harness_gateway_arn",
#     "cognito_user_pool_id", "cognito_client_id",
#     "gateway_m2m_client_id", "gateway_token_endpoint", "gateway_scope",
#     "gateway_m2m_client_secret",
#     "user_password",
#     "dd_api_key", "dd_app_key", "dd_site",
# ]
# for param in ssm_params:
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{param}")
#         print(f"  Deleted: {SSM_PREFIX}/{param}")
#     except ssm.exceptions.ParameterNotFound:
#         pass  # already gone — re-run safe
#     except Exception as e:
#         print(f"  Failed to delete {SSM_PREFIX}/{param}: {e}")
#
# print("\n✅ All pre-requisite resources cleaned up.")
